# register-back-fn-after-wrap — worked example 1: Register the Backward Function for torch.exp

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Every differentiable operation needs a backward function registered in the lookup table under the key `(fwd_fn, argnum)`. For `torch.exp`, the derivative of `exp(x)` with respect to `x` is `exp(x)` itself — equivalently, the forward output `out`. The two-step idiom is: (1) wrap the forward function, (2) register a back function at each differentiable argument position.

## Worked solution

**Step 1 — understand the math.** If `out = exp(x)`, then `d(out)/d(x) = exp(x) = out`. So the back function multiplies the incoming gradient by `out`: `grad_x = grad_out * out`.

**Step 2 — define the back function.** It receives `(grad_out, out, x)` by convention — the incoming gradient, the saved output, and the forward input(s). For exp we only need `out`.

**Step 3 — build the lookup table.** `BackwardFuncLookup` wraps a dict keyed by `(fwd_fn, argnum)` tuples. `add_back_func` stores an entry; `get_back_func` retrieves it.

**Step 4 — register.** Call `BACK_FUNCS.add_back_func(t.exp, 0, exp_back)`. Argnum 0 because `exp` takes one tensor argument at position 0.

**Step 5 — verify.** Retrieve the back fn via `get_back_func(t.exp, 0)` and call it. With `x = [1.0, 2.0]`, `out = exp(x)`, and `grad_out = ones`, we expect `grad = out`.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def exp_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    # d/dx exp(x) = exp(x) = out
    return grad_out * out

def register_exp(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.exp, 0, exp_back)

# --- exercise and print ---
BACK_FUNCS = BackwardFuncLookup()
register_exp(BACK_FUNCS)

x = t.tensor([1.0, 2.0, 3.0])
out = t.exp(x)
grad_out = t.ones_like(x)

back_fn = BACK_FUNCS.get_back_func(t.exp, 0)
grad_x = back_fn(grad_out, out, x)
print('x:       ', x)
print('out:     ', out)
print('grad_x:  ', grad_x)  # Should equal out
print('match:', t.allclose(grad_x, out))